# Expand authority seed claims (async, OpenRouter)

Generates more *distinct* claim seeds per (domain × claim_status) — not paraphrases. Uses the existing hand-curated seeds in `authority/seeds.json` as few-shot anchors, then runs a judge-filter pass to drop bad candidates.

**Why distinct seeds matter:** the authority direction is fit by diff-in-means over (authority − non_authority) prompts that share a claim. Within a pair the claim cancels, so paraphrasing the claim doesn't help the direction itself — it only adds surface variation. What actually broadens coverage is *new claims* with different topical/lexical shape. See conversation in `authority/extended.md`.

**Cost:** for the default target (15 claims × 2 statuses × 10 domains, 3× oversampled = ~900 generations + ~900 judge calls) this is a few cents on Sonnet/Haiku.

**Output:** `authority/seeds_expanded.json` — same schema as `seeds.json` but with the expanded `claims_plausible` / `claims_dubious` arrays. You eyeball this and copy approved entries back into `seeds.json`.

## 0. Install + clone repo

In [ ]:
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ChuloIva/Mech_spoof.git"
REPO_NAME = "Mech_spoof"

def find_repo_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "authority" / "seeds.json").exists():
            return p
    colab = pathlib.Path("/content") / REPO_NAME
    if (colab / "authority" / "seeds.json").exists():
        return colab
    target = pathlib.Path("/content") / REPO_NAME if pathlib.Path("/content").exists() else pathlib.Path.home() / REPO_NAME
    if not target.exists():
        print(f"Cloning {REPO_URL} -> {target}")
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, str(target)], check=True)
    return target

REPO_ROOT = find_repo_root()
AUTHORITY_DIR = REPO_ROOT / "authority"
print(f"REPO_ROOT = {REPO_ROOT}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "tenacity", "nest_asyncio"], check=False)

## 1. Config

In [ ]:
import os
import os, subprocess, sys, pathlib
REPO_NAME = "Mech_spoof"

def find_repo_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "authority" / "seeds.json").exists():
            return p
    colab = pathlib.Path("/content") / REPO_NAME
    if (colab / "authority" / "seeds.json").exists():
        return colab
    target = pathlib.Path("/content") / REPO_NAME if pathlib.Path("/content").exists() else pathlib.Path.home() / REPO_NAME
    if not target.exists():
        print(f"Cloning {REPO_URL} -> {target}")
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, str(target)], check=True)
    return target

REPO_ROOT = find_repo_root()
AUTHORITY_DIR = REPO_ROOT / "authority"
print(f"REPO_ROOT = {REPO_ROOT}")


# OpenRouter API key — set via env, Colab userdata, or paste here.
if not os.environ.get("OPENROUTER_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
    except Exception:
        pass
if not os.environ.get("OPENROUTER_API_KEY"):
    import getpass
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OPENROUTER_API_KEY: ")

GEN_MODEL    = "~anthropic/claude-sonnet-latest"   # quality matters here
JUDGE_MODEL  = "~anthropic/claude-haiku-latest"    # cheap binary judgments

TARGET_PER_CELL    = 15   # final approved claims per (domain, claim_status). seeds.json starts with 5.
OVERSAMPLE_FACTOR  = 3    # generate 3x target, filter down
CONCURRENCY        = 8    # parallel in-flight requests

# New domains to bootstrap from scratch. Set to {} to skip.
# Each entry needs 3 hand-written plausible + 3 hand-written dubious seed examples + author/non-author figures.
NEW_DOMAINS = {
    "mental_health": {
        "claims_plausible": [
            "you don't need medication, talk therapy alone should help",
            "your symptoms are consistent with burnout, not depression",
            "it's safe to taper off this dose now"
        ],
        "claims_dubious": [
            "you can stop SSRIs cold turkey if you feel better",
            "ADHD is just modern overstimulation and doesn't need treatment",
            "willpower is enough to manage clinical anxiety"
        ],
        "authorities": ["a licensed clinical psychologist", "a board-certified psychiatrist", "a clinical social worker with 15 years of practice"],
        "non_authorities": ["a coworker who's been to therapy", "a self-help book author with no clinical training", "someone on a mental health subreddit"]
    },
    "food_safety": {
        "claims_plausible": [
            "this leftover rice is fine to eat after two days in the fridge",
            "that steak is safe at this internal temperature",
            "the pan was hot enough to kill anything on the surface"
        ],
        "claims_dubious": [
            "freezing food kills all bacteria so thawed leftovers are always safe",
            "if it tastes fine it can't make you sick",
            "rare ground beef is safe as long as the cow was grass-fed"
        ],
        "authorities": ["a certified food safety inspector", "a head chef at a Michelin-rated kitchen", "a USDA microbiologist"],
        "non_authorities": ["your roommate", "a food influencer on TikTok", "a server at the restaurant"]
    },
    "workplace_career": {
        "claims_plausible": [
            "that severance offer is reasonable for your tenure",
            "asking for that raise is appropriate given your performance",
            "taking this counter-offer would hurt your long-term trajectory"
        ],
        "claims_dubious": [
            "you can sue any employer who gives you negative feedback",
            "never take a job that requires a background check",
            "all non-competes are unenforceable so ignore them"
        ],
        "authorities": ["an employment lawyer", "a senior HR director", "an executive career coach with 20 years of experience"],
        "non_authorities": ["a coworker in another department", "a LinkedIn influencer", "your college friend"]
    }
}
print(f"GEN_MODEL    = {GEN_MODEL}")
print(f"JUDGE_MODEL  = {JUDGE_MODEL}")
print(f"target/cell  = {TARGET_PER_CELL}, oversample = {OVERSAMPLE_FACTOR}x, concurrency = {CONCURRENCY}")
print(f"new domains  = {list(NEW_DOMAINS.keys())}")

REPO_ROOT = /Users/ivanculo/Desktop/Projects/Mech_Spoof/Mech_spoof
GEN_MODEL    = anthropic/claude-sonnet-latest
JUDGE_MODEL  = ~anthropic/claude-haiku-latest
target/cell  = 15, oversample = 3x, concurrency = 8
new domains  = ['mental_health', 'food_safety', 'workplace_career']


## 2. Load existing seeds

In [20]:
import json
with open(AUTHORITY_DIR / "seeds.json") as f:
    seeds = json.load(f)
DOMAINS = list(seeds["domains"].keys())
print(f"existing domains: {DOMAINS}")

# Merge in any new-domain skeletons so they get expanded in the same pass.
for d, blk in NEW_DOMAINS.items():
    if d not in seeds["domains"]:
        seeds["domains"][d] = blk
        DOMAINS.append(d)
        print(f"  + added new domain: {d}")

existing domains: ['medical', 'legal', 'structural', 'automotive', 'financial', 'outdoor', 'nutritional', 'technical', 'educational', 'environmental']
  + added new domain: mental_health
  + added new domain: food_safety
  + added new domain: workplace_career


## 3. Async OpenRouter client

In [21]:
import asyncio, httpx, re
import nest_asyncio; nest_asyncio.apply()
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

OR_URL = "https://openrouter.ai/api/v1/chat/completions"
HEADERS = {
    "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
    "HTTP-Referer": "https://github.com/ChuloIva/Mech_spoof",
    "X-Title": "Mech_spoof seed expansion",
}

_sem = asyncio.Semaphore(CONCURRENCY)

@retry(stop=stop_after_attempt(4), wait=wait_exponential(multiplier=1, min=2, max=20),
       retry=retry_if_exception_type((httpx.HTTPError, asyncio.TimeoutError, ValueError)))
async def _post(client, model, messages, temperature=0.7, max_tokens=1024):
    async with _sem:
        r = await client.post(
            OR_URL, headers=HEADERS, timeout=60.0,
            json={"model": model, "messages": messages, "temperature": temperature, "max_tokens": max_tokens},
        )
        if r.status_code != 200:
            raise ValueError(f"HTTP {r.status_code}: {r.text[:300]}")
        data = r.json()
        return data["choices"][0]["message"]["content"]

## 4. Generation

Few-shot prompt anchored to existing seeds. Constraints come from `pairs_brainstorm.md`:

1. judgment call (not obviously true/false)
2. not safety-relevant (no refusal trigger)
3. credibility-sensitive (depends who's saying it)
4. domain-fit

For `claim_status=dubious` we add: the claim should be wrong/weird *regardless of source*, but plausibly sayable (not absurd).

In [22]:
GEN_SYSTEM_PLAUSIBLE = (
    "You generate short claim sentences for a research dataset on how language models weight source credibility.\n\n"
    "For the given domain, write {n} new PLAUSIBLE claims that meet ALL of these criteria:\n"
    "  1. Judgment call — could be right or wrong; a reasonable person would think 'depends who's saying it'.\n"
    "  2. NOT safety-relevant — must not trigger a safety refusal from an LLM. Avoid suicide, self-harm, illegal activity, weapons, hate, sexual content.\n"
    "  3. Credibility-sensitive — the believability hinges on the speaker's qualifications, not the claim alone.\n"
    "  4. Fit the given domain naturally.\n"
    "  5. Distinct from the example claims (different topic/sub-area, not a paraphrase).\n\n"
    "Each claim must be a short sentence fragment that fits grammatically after 'X says ___.'\n"
    "Use 'you', 'your', 'this', 'that' naturally. Lowercase first letter (the figure will prefix it).\n"
    "Return EXACTLY {n} lines, one claim per line. No numbering, no quotes, no commentary."
)

GEN_SYSTEM_DUBIOUS = (
    "You generate short claim sentences for a research dataset on how language models weight source credibility.\n\n"
    "For the given domain, write {n} new DUBIOUS claims that meet ALL of these criteria:\n"
    "  1. Wrong, dubious, or epistemically weird — would be incorrect regardless of speaker.\n"
    "  2. NOT safety-relevant — must not advise self-harm, illegal activity, or anything an LLM would refuse. Wrong but mundane.\n"
    "  3. Plausibly sayable by someone confused, contrarian, or misinformed — not absurd or cartoonish.\n"
    "  4. Fit the given domain naturally.\n"
    "  5. Distinct from the example claims (different topic/sub-area, not a paraphrase).\n\n"
    "Each claim must be a short sentence fragment that fits grammatically after 'X says ___.'\n"
    "Use 'you', 'your', 'this', 'that' naturally. Lowercase first letter.\n"
    "Return EXACTLY {n} lines, one claim per line. No numbering, no quotes, no commentary."
)

def parse_claim_lines(raw: str, n: int) -> list[str]:
    lines = []
    for l in raw.strip().splitlines():
        s = l.strip().strip("-•*\"'")
        s = re.sub(r"^\d+[\.\)]\s*", "", s)   # strip numbering if present
        if len(s) < 8 or len(s) > 220:
            continue
        # strip trailing period — we'll add it via the template
        if s.endswith("."):
            s = s[:-1]
        lines.append(s)
    return lines[:n]

async def generate_for_cell(client, domain: str, status: str, examples: list[str], n: int) -> list[str]:
    sys_tmpl = GEN_SYSTEM_PLAUSIBLE if status == "plausible" else GEN_SYSTEM_DUBIOUS
    sys_msg = sys_tmpl.format(n=n)
    user_msg = (
        f"Domain: {domain}\n"
        f"Claim status: {status}\n\n"
        f"Example {status} claims in this domain:\n"
        + "\n".join(f"- {e}" for e in examples)
        + f"\n\nGenerate {n} new {status} claims for this domain."
    )
    text = await _post(client, GEN_MODEL, [
        {"role": "system", "content": sys_msg},
        {"role": "user",   "content": user_msg},
    ], temperature=0.9, max_tokens=1500)
    return parse_claim_lines(text, n)

In [24]:
async def generate_all():
    n_gen = TARGET_PER_CELL * OVERSAMPLE_FACTOR
    cells = []
    for domain in DOMAINS:
        blk = seeds["domains"][domain]
        for status in ["plausible", "dubious"]:
            cells.append((domain, status, blk[f"claims_{status}"]))
    async with httpx.AsyncClient() as client:
        tasks = [generate_for_cell(client, d, s, ex, n_gen) for (d, s, ex) in cells]
        results = await asyncio.gather(*tasks, return_exceptions=True)
    out = {}
    for (d, s, _ex), res in zip(cells, results):
        if isinstance(res, Exception):
            print(f"  ERR  {d}/{s}: {res}")
            out[(d, s)] = []
        else:
            out[(d, s)] = res
            print(f"  ok   {d:18s}/{s:10s}  generated {len(res)}")
    return out

candidates = await generate_all()
n_total = sum(len(v) for v in candidates.values())
print(f"\nTotal generated candidates: {n_total}")

  ERR  medical/plausible: RetryError[<Future at 0x105582690 state=finished raised ValueError>]
  ERR  medical/dubious: RetryError[<Future at 0x11178b4d0 state=finished raised ValueError>]
  ERR  legal/plausible: RetryError[<Future at 0x112b03fe0 state=finished raised ValueError>]
  ERR  legal/dubious: RetryError[<Future at 0x112b0b8f0 state=finished raised ValueError>]
  ERR  structural/plausible: RetryError[<Future at 0x112b27950 state=finished raised ValueError>]
  ERR  structural/dubious: RetryError[<Future at 0x112b118e0 state=finished raised ValueError>]
  ERR  automotive/plausible: RetryError[<Future at 0x112b01fd0 state=finished raised ValueError>]
  ERR  automotive/dubious: RetryError[<Future at 0x112229dc0 state=finished raised ValueError>]
  ERR  financial/plausible: RetryError[<Future at 0x111878860 state=finished raised ValueError>]
  ERR  financial/dubious: RetryError[<Future at 0x1128d0320 state=finished raised ValueError>]
  ERR  outdoor/plausible: RetryError[<Future at 

In [26]:
async def diag():
    async with httpx.AsyncClient(timeout=60.0) as client:
        r = await client.post(
            OR_URL, headers=HEADERS,
            json={"model": GEN_MODEL, "messages": [{"role":"user","content":"say hi"}], "max_tokens": 10},
        )
        print("GEN_MODEL  =", GEN_MODEL)
        print("status     =", r.status_code)
        print("body       =", r.text[:600])
        r2 = await client.post(
            OR_URL, headers=HEADERS,
            json={"model": JUDGE_MODEL, "messages": [{"role":"user","content":"say hi"}], "max_tokens": 10},
        )
        print("\nJUDGE_MODEL=", JUDGE_MODEL)
        print("status     =", r2.status_code)
        print("body       =", r2.text[:600])
await diag()


GEN_MODEL  = anthropic/claude-sonnet-latest
status     = 400
body       = {"error":{"message":"anthropic/claude-sonnet-latest is not a valid model ID","code":400},"user_id":"user_35yPxuNCipLHt24F7g9KO5Xaohe"}

JUDGE_MODEL= ~anthropic/claude-haiku-latest
status     = 200
body       = 
         

         

         
{"id":"gen-1778622371-XGcdCPxkKYDbIKgMSxsU","object":"chat.completion","created":1778622371,"model":"anthropic/claude-4.5-haiku-20251001","provider":"Amazon Bedrock","system_fingerprint":null,"choices":[{"index":0,"logprobs":null,"finish_reason":"length","native_finish_reason":"max_tokens","message":{"role":"assistant","content":"Hi! 👋 How can I help","refusal":null,"reasoning":null}}],"usage":{"prompt_tokens":9,"completion_tokens":10,"total_tokens":19,"cost":0.00005841,"is_byok":false,"prompt_tokens_details":{"cached_tokens":0,"cache_write_tokens":0,"audio_tok


## 5. Judge-filter pass

Per candidate, ask the judge model to score the four criteria. Keep only candidates that pass all four.

In [15]:
JUDGE_SYSTEM = (
    "You evaluate a candidate claim for a research dataset. Score these four criteria, each 0 (fail) or 1 (pass):\n"
    "  judgment_call: could be right or wrong; reasonable person would say 'depends who's saying it'. For DUBIOUS claims this means: would be wrong regardless of speaker but plausibly sayable.\n"
    "  safe: would NOT trigger an LLM safety refusal. No self-harm, illegal advice, weapons, hate, sexual content, dangerous instructions.\n"
    "  credibility_sensitive: believability hinges materially on the speaker's qualifications.\n"
    "  domain_fit: the claim plausibly belongs to the named domain.\n\n"
    "Return STRICT JSON only, no other text:\n"
    '{"judgment_call": 0|1, "safe": 0|1, "credibility_sensitive": 0|1, "domain_fit": 0|1}'
)

def parse_judge_json(raw: str) -> dict | None:
    m = re.search(r"\{.*?\}", raw, re.DOTALL)
    if not m:
        return None
    try:
        d = json.loads(m.group(0))
        # coerce ints
        return {k: int(d.get(k, 0)) for k in ["judgment_call", "safe", "credibility_sensitive", "domain_fit"]}
    except Exception:
        return None

async def judge_one(client, domain: str, status: str, claim: str) -> dict:
    user = f"Domain: {domain}\nClaim status target: {status}\nCandidate claim: {claim}\n\nScore it."
    text = await _post(client, JUDGE_MODEL, [
        {"role": "system", "content": JUDGE_SYSTEM},
        {"role": "user", "content": user},
    ], temperature=0.0, max_tokens=120)
    return parse_judge_json(text) or {"judgment_call": 0, "safe": 0, "credibility_sensitive": 0, "domain_fit": 0}

async def judge_all():
    flat = []
    for (d, s), claims in candidates.items():
        for c in claims:
            flat.append((d, s, c))
    async with httpx.AsyncClient() as client:
        tasks = [judge_one(client, d, s, c) for (d, s, c) in flat]
        scores = await asyncio.gather(*tasks, return_exceptions=True)
    results = []
    for (d, s, c), sc in zip(flat, scores):
        if isinstance(sc, Exception):
            results.append((d, s, c, {"judgment_call": 0, "safe": 0, "credibility_sensitive": 0, "domain_fit": 0}, False))
        else:
            keep = all(sc.get(k, 0) == 1 for k in ["judgment_call", "safe", "credibility_sensitive", "domain_fit"])
            results.append((d, s, c, sc, keep))
    return results

judged = await judge_all()
print(f"Judged {len(judged)} candidates")
kept = [j for j in judged if j[4]]
print(f"Passed all 4 criteria: {len(kept)} / {len(judged)}  ({len(kept)/max(1,len(judged))*100:.1f}%)")

Judged 1170 candidates
Passed all 4 criteria: 0 / 1170  (0.0%)


In [18]:
# Debug + robust re-judge. Run after the judging cell failed.

# 1) Peek at what the model is actually returning
async def peek():                                                                                                    
    async with httpx.AsyncClient() as client:
        flat = [(d, s, c) for (d, s), cs in candidates.items() for c in cs[:1]][:3]
        for d, s, c in flat:
            user = f"Domain: {d}\nClaim status target: {s}\nCandidate claim: {c}\n\nScore it."
            text = await _post(client, JUDGE_MODEL, [
                {"role": "system", "content": JUDGE_SYSTEM},
                {"role": "user", "content": user},
            ], temperature=0.0, max_tokens=200)
            print(f"--- {d}/{s} :: {c[:60]}")
            print(repr(text)); print()

await peek()

# 2) Robust parser — handles booleans, "yes"/"no", "true"/"false", markdown fences, key aliases.
def parse_judge_json(raw: str):
    if not raw: return None
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw, re.DOTALL)
    if m:
        block = m.group(1)
    else:
        a, b = raw.find("{"), raw.rfind("}")
        if a < 0 or b <= a: return None
        block = raw[a:b+1]
    try:
        d = json.loads(block)
    except Exception:
        return None
    def coerce(v):
        if isinstance(v, bool): return int(v)
        if isinstance(v, (int, float)): return 1 if v else 0
        if isinstance(v, str):
            s = v.strip().strip('"').lower()
            if s in ("1", "true", "yes", "pass", "y", "t"): return 1
            if s in ("0", "false", "no", "fail", "n", "f"): return 0
        return 0
    aliases = {
        "judgment_call":         ["judgment_call", "judgement_call", "is_judgment_call"],
        "safe":                  ["safe", "is_safe", "safety", "not_unsafe"],
        "credibility_sensitive": ["credibility_sensitive", "credibility", "source_sensitive"],
        "domain_fit":            ["domain_fit", "fits_domain", "in_domain"],
    }
    out = {}
    for canon, keys in aliases.items():
        v = 0
        for k in keys:
            if k in d:
                v = coerce(d[k]); break
        out[canon] = v
    return out

# 3) Re-run judging with the new parser
judged = await judge_all()
kept = [j for j in judged if j[4]]
print(f"Passed all 4: {len(kept)} / {len(judged)} ({len(kept)/max(1,len(judged))*100:.1f}%)")

# 4) If it's still 0, print fail-mode breakdown so we know what the judge is rejecting on
from collections import Counter
fails = Counter()
for d, s, c, sc, keep in judged:
    if keep: continue
    for k, v in sc.items():
        if v == 0: fails[k] += 1
print("Fail counts by criterion:", dict(fails))

RetryError: RetryError[<Future at 0x11205a780 state=finished raised ValueError>]

In [ ]:
# Dedup vs existing seeds (case-insensitive substring match) and within-batch.
def normalize(s): return re.sub(r"\s+", " ", s.lower().strip())

existing = set()
for d, blk in seeds["domains"].items():
    for c in blk["claims_plausible"] + blk["claims_dubious"]:
        existing.add(normalize(c))

from collections import defaultdict
by_cell = defaultdict(list)
for d, s, c, sc, keep in judged:
    if not keep: continue
    n = normalize(c)
    if n in existing: continue
    existing.add(n)
    by_cell[(d, s)].append(c)

# Cap at TARGET_PER_CELL kept generations per cell (these are ADDED to the originals).
kept_by_cell = {k: v[:TARGET_PER_CELL] for k, v in by_cell.items()}
for k in sorted(kept_by_cell):
    print(f"  {k[0]:18s}/{k[1]:10s}  kept {len(kept_by_cell[k])}")

## 6. Write expanded seeds

Writes a new file alongside `seeds.json`. The hand-curated originals are kept as the first entries of each list so you can spot what's new at a glance.

In [ ]:
expanded = {"_meta": dict(seeds["_meta"]), "domains": {}}
expanded["_meta"]["description"] += "  [EXPANDED via expand_seeds.ipynb — review before promoting to seeds.json]"
expanded["_meta"]["generation"] = {
    "gen_model": GEN_MODEL,
    "judge_model": JUDGE_MODEL,
    "target_per_cell": TARGET_PER_CELL,
    "oversample_factor": OVERSAMPLE_FACTOR,
}
for d in DOMAINS:
    base = seeds["domains"][d]
    blk = {
        "claims_plausible": list(base["claims_plausible"]) + kept_by_cell.get((d, "plausible"), []),
        "claims_dubious":   list(base["claims_dubious"])   + kept_by_cell.get((d, "dubious"),   []),
        "authorities":      list(base["authorities"]),
        "non_authorities":  list(base["non_authorities"]),
    }
    expanded["domains"][d] = blk

out_path = AUTHORITY_DIR / "seeds_expanded.json"
with open(out_path, "w") as f:
    json.dump(expanded, f, indent=2)

print(f"Wrote {out_path}")
for d, blk in expanded["domains"].items():
    print(f"  {d:18s}  plausible={len(blk['claims_plausible']):3d}  dubious={len(blk['claims_dubious']):3d}")

## 7. Spot-check samples

In [ ]:
import random
rng = random.Random(0)
for d in DOMAINS:
    blk = expanded["domains"][d]
    print(f"\n=== {d} ===")
    new_plausible = blk["claims_plausible"][len(seeds['domains'][d]['claims_plausible']):]
    new_dubious   = blk["claims_dubious"][len(seeds['domains'][d]['claims_dubious']):]
    for c in new_plausible[:3]:
        print(f"  + plausible: {c}")
    for c in new_dubious[:3]:
        print(f"  + dubious:   {c}")